<a href="https://colab.research.google.com/github/HiveCase/DLP-DA5013/blob/main/Deep_Learning_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## L8: Building & training a tokenizer: byte-pair encoding with bookcorpus

In [3]:
# !pip install -U "datasets==3.6.0"

In [4]:
from datasets import load_dataset

In [1]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import Lowercase

In [7]:
from pprint import pprint

In [8]:
ds = load_dataset('bookcorpus/bookcorpus', split='all')
pprint(ds)

Dataset({
    features: ['text'],
    num_rows: 74004228
})


In [9]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
tokenizer.normalizer = Lowercase()

In [10]:
from tokenizers.trainers import BpeTrainer

In [11]:
trainer = BpeTrainer(vocab_size=32000, special_tokens=["[PAD]","[UNK]"], continuing_subword_prefix="##")

In [12]:
# batch tokenization function
def get_examples(batch_size=1000):
  for i in range(0, len(ds), batch_size):
    yield ds[i:i+batch_size]['text']

In [13]:
from multiprocessing import cpu_count

In [14]:
print(cpu_count())

2


In [15]:
tokenizer.train_from_iterator(get_examples(batch_size=10000), trainer=trainer, length=len(ds))

In [19]:
import os

os.makedirs("model", exist_ok=True)

tokenizer.model.save("model", prefix="hopper")

['model/hopper-vocab.json', 'model/hopper-merges.txt']

In [20]:
with open('model/hopper-merges.txt', 'r') as file:
  row = 0
  num_lines = 10
  for line in file.readlines():
    print(line)
    row+=1
    if row>=num_lines:
      break

#version: 0.2

##h ##e

t ##he

##i ##n

##e ##r

##e ##d

##o ##u

##n ##d

##in ##g

t ##o



In [21]:
with open('model/hopper-merges.txt', 'r') as file:
  row = 0
  num_lines = 10
  for line in reversed(file.readlines()):
    print(line)
    row+=1
    if row>=num_lines:
      break

mel ##anthe

black ##er

ad ##ject

v ##ang

betroth ##al

tiptoe ##ing

restroom ##s

consol ##ing

esp ##ionage

influ ##x



In [22]:
with open('model/hopper-merges.txt', 'r') as file:
  lines = file.readlines()
print(f"Number of merges: {len(lines)}")

Number of merges: 31871


## L9: Encdode & decoder | tokenization explained with BERT, padding & truncation